# Fig. 7g: information capacity of the ground truth, of the 'good' sorted units and of the degraded sorted units (evoked model)

**author**: laquitainesteeve@gmail.com

**purpose**: reproduce panel g of the paper: manifold information capacity, divided by its lower bound, of the ground truth neurons (grey), of the sorted units of each spike sorter that include the 'good', 'merged' and 'partial' units (red; pink in Fig. 5d) and of the degraded units, 'merged' and 'partial' only (white; Fig. 5c,d). The distribution of the unit types is matched between the red and the white groups. Source data and svg are saved in figures/8_source_data/fig7g/.

**method (as in the paper's notebook fig7_G_capacity_vs_quality_revised.ipynb)**
* the capacity of each population is computed on bootstraps of the units (population responses to the 360 stimulus directions, 200 latents, `neural_manifolds_replicaMFT`); chance is 1
* the bootstrapped capacities are read from the csv file of the paper (`igeometrics_by_sorter_and_quality_00.csv`); they are **not recomputed** here (the capacity analysis needs the `mftma` package)
* the file has 200 bootstrapped capacities for the ground truth, 200 for the 'good + merged + partial' units of each sorter and **400** for its degraded units (two sets of 200 with different medians); as in the paper's notebook, the plot pools the 400 values of the degraded units
* box plots (notch); red dashed line: mean capacity of the ground truth; blue dashed line: chance (1)
* comparison with the ground truth: two-sample t-test between the bootstrapped capacities (as in the paper's notebook)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import spikeinterface as si
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# set project path
PROJ_PATH = "/home/steeve/steeve/epfl/code/spikebias/"
os.chdir(PROJ_PATH)

# custom package
import seaborn as sns
from scipy.stats import ttest_ind

# paths (bootstrapped manifold capacities of the ground truth and of the sorted populations, evoked simulation, 1 hour)
NEURAL_CODE_PATH = "dataset/01_intermediate/neural_code/npx_evoked/"
IGEOM_S_Q = NEURAL_CODE_PATH + "igeometrics_by_sorter_and_quality_00.csv"

# FIGURE SETTINGS
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["axes.linewidth"] = 0.5
plt.rcParams["axes.spines.top"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["xtick.minor.size"] = 2 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.minor.size"] = 2 * 1.1
legend_cfg = {"frameon": False, "handletextpad": 0.5}

# save config
savefig_cfg = {"transparent": True, "dpi": 300}

In [ ]:
# Nature figure guidelines for every saved figure: Arial, 5-7 pt, editable text, images of at least 300 dpi, a .pdf next to each .svg
import os, sys
sys.path.insert(0, os.getcwd())     # the repository (the notebook moved there above)
from src.nodes import figure_style
figure_style.apply()

## Load the bootstrapped capacities

In [2]:
%%time

igeom_df = pd.read_csv(IGEOM_S_Q)
SORTERS = ["KS4", "KS3", "KS2.5", "KS2", "KS", "HS"]
CLASSES = {"ground_truth": "ground truth", "mixed: good + overmerger + oversplitter": "good + merged + partial", "mixed: overmerger + oversplitter": "degraded (merged + partial)"}
df = igeom_df[["Sorter", "Unit class", "Capacity"]].copy()
df["population"] = df["Sorter"].replace({"ground_truth": "ground truth"})
df["unit_class"] = df["Unit class"].map(CLASSES)
df["bootstrap"] = df.groupby(["Sorter", "Unit class"]).cumcount()
df = df.rename(columns={"Capacity": "capacity_over_lower_bound"})[["population", "unit_class", "bootstrap", "capacity_over_lower_bound"]]
print(df.groupby(["population", "unit_class"], sort=False)["capacity_over_lower_bound"].agg(["count", "median"]).round(3))

                                          count  median
population   unit_class                                
ground truth ground truth                   200   1.249
KS4          good + merged + partial        200   1.168
             degraded (merged + partial)    400   1.149
KS3          good + merged + partial        200   1.131
             degraded (merged + partial)    400   1.096
KS2.5        good + merged + partial        200   1.153
             degraded (merged + partial)    400   1.143
KS2          good + merged + partial        200   1.145
             degraded (merged + partial)    400   1.150
KS           good + merged + partial        200   1.193
             degraded (merged + partial)    400   1.010
HS           good + merged + partial        200   1.160
             degraded (merged + partial)    400   1.139
CPU times: user 4.92 ms, sys: 3 ms, total: 7.92 ms
Wall time: 7.92 ms


## Capacity of the sorted units against the ground truth and the 'good' units

In [3]:
gt = df[df["population"] == "ground truth"]["capacity_over_lower_bound"]
rows = []
for sorter in SORTERS:
    good = df[(df["population"] == sorter) & (df["unit_class"] == "good + merged + partial")]["capacity_over_lower_bound"]
    bad = df[(df["population"] == sorter) & (df["unit_class"] == "degraded (merged + partial)")]["capacity_over_lower_bound"]
    t_good, p_good = ttest_ind(good, gt)
    rows.append(dict(population=sorter, median_capacity_good=good.median(), median_capacity_degraded=bad.median(),
                     good_vs_ground_truth_percent=100 * (good.median() - gt.median()) / (gt.median() - 1), t_good_vs_ground_truth=t_good, p_good_vs_ground_truth=p_good,
                     degraded_vs_good_percent=100 * (bad.median() - good.median()) / (good.median() - 1)))
stats = pd.DataFrame(rows)
display(stats.round(4))
print("good units against the ground truth: average change of the capacity above the lower bound (%):", round(stats["good_vs_ground_truth_percent"].mean(), 1),
      "| t from", round(stats["t_good_vs_ground_truth"].min(), 2), "to", round(stats["t_good_vs_ground_truth"].max(), 2))

,population,median_capacity_good,median_capacity_degraded,good_vs_ground_truth_percent,t_good_vs_ground_truth,p_good_vs_ground_truth,degraded_vs_good_percent
0,KS4,1.1681,1.1487,-32.4320,-22.5816,0.0,-11.5415
1,KS3,1.1307,1.0965,-47.4362,-31.3536,0.0,-26.1941
2,KS2.5,1.1532,1.1433,-38.3975,-24.5415,0.0,-6.4887
3,KS2,1.1450,1.1499,-41.6868,-28.3070,0.0,3.3248
4,KS,1.1933,1.0104,-22.2622,-15.5986,0.0,-94.6309
5,HS,1.1603,1.1388,-35.5401,-24.8421,0.0,-13.4323


good units against the ground truth: average change of the capacity above the lower bound (%): -36.3 | t from -31.35 to -15.6


## Plot, same as the paper's panel g

In [4]:
# paper geometry (page 22 of the paper): canvas 124.6 x 117.0 pt, Arial 5 pt text, editable (src/nodes/validation/capacity_plots.py)
from src.nodes.validation.capacity_plots import plot_quality_capacity

def plot_source_data(df: pd.DataFrame):
    return plot_quality_capacity(df)

## Plot and save the figure and the source data

In [5]:
%%time

# plot
fig = plot_source_data(df)

# save figure
os.makedirs("figures/8_source_data/fig7g", exist_ok=True)
plt.savefig("figures/8_source_data/fig7g/fig7g.svg", transparent=True)   # the .pdf is written next to the .svg

# save source data (sheet 1: bootstrapped capacities; sheet 2: comparison with the ground truth and between unit classes)
with pd.ExcelWriter("figures/8_source_data/fig7g/fig7g.xlsx") as writer:
    df.to_excel(writer, sheet_name="capacity_bootstraps", index=False)
    stats.to_excel(writer, sheet_name="vs_ground_truth", index=False)

## Check: replot from the saved source data

In [6]:
%%time

df = pd.read_excel("figures/8_source_data/fig7g/fig7g.xlsx", sheet_name="capacity_bootstraps")
fig = plot_source_data(df)